# Pure discontinuous test case

## Initialize


### For parallel testing 

Create a serveur controlling  n>1 mpi engines
(consider openmpi as mpi backend)

In [ ]:
import ipyparallel as ipp
cluster = ipp.Cluster(engines="mpi", n=1)
rc = cluster.start_and_connect_sync()
view=rc[:]

In [ ]:
%pxconfig --progress-after -1

In [ ]:
%%px --local
from dolfinx import cpp
from dolfinx import log
from dolfinx import mesh
from dolfinx import fem
from dolfinx import plot
import ufl
import petsc4py
from dolfinx.fem import petsc
import numpy as np
from mpi4py import MPI
from mpi4py.typing import InPlace

import pyvista as pv
import basix
from importlib import reload, import_module
from dolfinx_mpc import MultiPointConstraint
from dolfinx_mpc import LinearProblem as LinearProblem_mpc

#log.set_log_level(cpp.log.LogLevel.INFO)

### Import twosacle library

In [ ]:
%%px --local
import sys
sys.dont_write_bytecode = True

In [ ]:
%%px --local
from twoscale import core
from twoscale import linear
from twoscale import util

In [ ]:
%%px --local
D2d=True
#D2d=False
D1d=True
D1d=False
s=2
ref=5
itmax=30
epsr=1.e-7
imp=0.01
petsc_options={"ksp_type":"preonly","pc_type":"lu","pc_factor_mat_solver_type":"mumps"}
petsc_options={"ksp_type":"cg","ksp_rtol":"1e-12","pc_type":"hypre","pc_hypre_type":"boomeramg","pc_hypre_type":"boomeramg","pc_hypre_boomeramg_coarsen_type":"HMIS","pc_hypre_boomeramg_relax_type_up":"l1scaled-SOR/Jacobi","pc_hypre_boomeramg_relax_type_down": "l1scaled-SOR/Jacobi","pc_hypre_boomeramg_relax_type_coarse":"Gaussian-elimination","pc_hypre_boomeramg_interp_type": "ext+i","pc_hypre_boomeramg_numfunctions": "2","pc_hypre_boomeramg_agg_nl": "0","pc_hypre_boomeramg_strong_threshold": "0.25","pc_hypre_boomeramg_print_statistics":"1"}


### Bacis test case 


In [ ]:
%%px --local
B=10.
L=20.
H=5.
if D1d:
    domain= mesh.create_interval(MPI.COMM_WORLD,2*s,[0.0,L],ghost_mode=mesh.GhostMode.none)
    odomain= mesh.create_interval(MPI.COMM_WORLD,2*s,[0.0, L],ghost_mode=mesh.GhostMode.none)    
else:
    if D2d:
        nd=[2*s, s]
        domain= mesh.create_rectangle(MPI.COMM_WORLD,[[0.0, -B / 2], [L, B / 2]],nd,ghost_mode=mesh.GhostMode.none)
        odomain= mesh.create_rectangle(MPI.COMM_WORLD,[[0.0, -B / 2], [L, B / 2]],nd,ghost_mode=mesh.GhostMode.none)
    else:
        nd=[2*s, 2*s, s]
        domain= mesh.create_box(MPI.COMM_WORLD,[[0.0, -B / 2, -H / 2], [L, B / 2, H / 2]],nd,ghost_mode=mesh.GhostMode.none)
        odomain= mesh.create_box(MPI.COMM_WORLD,[[0.0, -B / 2, -H / 2], [L, B / 2, H / 2]],nd,ghost_mode=mesh.GhostMode.none)
dim=domain.topology.dim

In [ ]:
%%px
pv.set_jupyter_backend("static")

In [ ]:
pv.set_jupyter_backend("static")
#pv.set_jupyter_backend("trame")


In [ ]:
%%px --local
pvopt1D={"show_edges":True, "show_scalar_bar":True,"point_size":4,"show_vertices":True}
if D1d:
    pvopt=pvopt1D
else:
    pvopt={"show_edges":True, "show_scalar_bar":True}

In [ ]:
#%%px --local
cells, types, x = plot.vtk_mesh(domain)
grid = pv.UnstructuredGrid(cells, types, x)
plt=pv.Plotter()
plt.add_mesh(grid,**pvopt)
plt.show()

In [ ]:
%%px 
grid=util.root_rank(domain)
if domain.comm.rank == 0:
    plt=pv.Plotter()
    plt.add_mesh(grid,**pvopt)
    plt.show()

In [ ]:
%px domain.name="test"

### Choose a mesh refinement localization criterion

finest mesh close to the middle.
The middle is fixed by posmx and is shifted from $\frac{L}{2}$ true middle position

In [ ]:
%%px --local
posmx=L/2+L/(10*s)

def crit1(coords,level):
    return np.logical_and(coords[0]<posmx+L/(8*s),coords[0]>posmx-L/(8*s))

### Choose macro node to enrich
select central region around $x=\frac{L}{2} \pm \frac{L}{8}$

In [ ]:
%%px --local
def enriched(coords):
    return np.logical_and(coords[0]>=3*L/8, coords[0]<=5*L/8)


## Scale Jump construction

In [ ]:
j=core.topDown(domain,enriched,crit1,ref)

In [ ]:
%%px
j=core.topDown(domain,enriched,crit1,ref)

In [ ]:
%%px --local
fdomain=j.getFineMesh
cdomain=j.getCoarseMesh

In [ ]:
fcells, ftypes, fx = plot.vtk_mesh(fdomain)
fgrid = pv.UnstructuredGrid(fcells, ftypes, fx)
plt=pv.Plotter(shape=(1, 3))
plt.subplot(0, 0)
plt.add_mesh(fgrid,**pvopt)
#pv.Plotter.add_mesh_clip_box(plt,fgrid)
idxmapc0=cdomain.topology.index_map(0)
xc=cdomain.geometry.x
idxmapcg=cdomain.geometry.index_map()
label=j.getEnriched
loc=np.array(label,dtype=int)
label=idxmapc0.local_to_global(loc)
xi=idxmapcg.global_to_local(label)
xl=xc[xi]
labele=j.getExtraEnriched
loc=np.array(labele,dtype=int)
labele=idxmapc0.local_to_global(loc)
xi=idxmapcg.global_to_local(labele)
xle=xc[xi]
plt.subplot(0, 1)
ccells, ctypes, cx = plot.vtk_mesh(cdomain)
cgrid = pv.UnstructuredGrid(ccells, ctypes, cx)
plt.add_mesh(cgrid,**pvopt)
plt.add_point_labels(xl, label, point_size=2, font_size=12, shape_color='red')
plt.add_point_labels(xle, labele, point_size=2, font_size=12, shape_color='orange')
plt.subplot(0, 2)
ocells, otypes, ox = plot.vtk_mesh(odomain)
ogrid = pv.UnstructuredGrid(ocells, otypes, ox)
plt.add_mesh(ogrid,**pvopt)

plt.show()

In red enriched node asked. In orange extra enriched node related to mesh transition

In [ ]:
%%px 
gridf=util.root_rank(fdomain)
gridc=util.root_rank(cdomain)
grido=util.root_rank(odomain)
if fdomain.comm.rank == 0:
    plt=pv.Plotter(shape=(1, 3))
    plt.subplot(0, 0)
    plt.add_mesh(gridf,**pvopt)
    plt.add_title("mixed (fine scale)\npartitions",font_size=15)
    plt.subplot(0, 1)
    plt.add_mesh(gridc,**pvopt)
    plt.add_title("coarse (balanced)\npartitions",font_size=15)
    plt.subplot(0, 2)
    plt.add_mesh(grido,**pvopt)
    plt.add_title("coarse (original)\npartitions",font_size=15)
    plt.show()

## Damage 

A damage field is use to impose a pure discontinuous displacement in the middle (i.e. posmx). In between $x=posmx \pm \Delta$ damage is one an is null otherwise. Thus only fine scale is able to get element fully damaged and have discontinuous behavior.

In [ ]:
%%px --local
delta=2*L/(7*s*ref)
def dam(x):
    res=np.zeros(len(x[0]))
    disco=np.logical_and(np.greater(x[0],posmx-delta),np.less(x[0],posmx+delta))
    res[disco]=0.999
    return res

damage space and field

In [ ]:
%%px --local
if D2d:
    element_scal = basix.ufl.element("Lagrange", "triangle", 1)
else:
   element_scal = basix.ufl.element("Lagrange", "tetrahedron", 1)

space_scal = fem.functionspace(fdomain, element_scal)
space_scalc = fem.functionspace(cdomain, element_scal)
d=fem.Function(space_scal,name="d")
d.interpolate(dam)
dc=fem.Function(space_scalc,name="dc")
dc.interpolate(dam)

In [ ]:
%%px
damf=util.root_field(d,1.,1)
damc=util.root_field(dc,1.,1)
if cdomain.comm.rank==0:
    plt=pv.Plotter(shape=(2, 1))
    plt.subplot(0, 0)
    plt.add_mesh(damf,**pvopt)
    plt.add_title("Damage field on fine scale",font_size=7)
    plt.subplot(1, 0)
    plt.add_mesh(damc,**pvopt)
    plt.add_title("Damage field on coarse scale",font_size=7)
    plt.show()

## Linear problem directely on fine mesh

### Simple elasticity problem description

#### Space

In [ ]:
%%px --local
space = fem.functionspace(fdomain, fdomain.ufl_domain().ufl_coordinate_element())

#### Lamè

In [ ]:
%%px --local
E=5.e+6
nu=0.3
mu=fem.Constant(fdomain,E / (2.0 * (1.0 + nu)))
lmbda = fem.Constant(fdomain,E * nu / ((1.0 + nu) * (1.0 - 2.0 * nu)))

#### Formulation

In [ ]:
%%px --local
u = ufl.TrialFunction(space)
v = ufl.TestFunction(space)

In [ ]:
%%px --local
def eps(v):
    return 0.5*(ufl.grad(v) + ufl.grad(v).T)

In [ ]:
%%px --local
def sigma(strain): 
    return (1-d)*(2.0*mu*strain + lmbda*ufl.tr(strain)*ufl.Identity(dim))

In [ ]:
%%px --local
a_ufl=ufl.inner(sigma(eps(u)), eps(v)) * ufl.dx
f=fem.Function(space,name='load')
f.x.array[:]=0.
b_ufl=ufl.dot(f,v)*ufl.dx

### Dirichelet

In [ ]:
%%px --local
if D1d:
    clamp=np.array([0.])
    loading=np.array([imp])
else:
    if D2d:
        clamp=np.array([0.,0.])
        loading=np.array([imp,0.])
    else:
        clamp=np.array([0.,0.,0.])
        loading=np.array([imp,0.,0.])
bc_nodes=mesh.locate_entities(fdomain, 0, lambda x: np.isclose(x[0], 0.))
bdofs=fem.locate_dofs_topological(space,0,bc_nodes)
bcb=fem.dirichletbc(clamp,bdofs,space)
imp_nodes=mesh.locate_entities(fdomain, 0, lambda x: np.isclose(x[0], L))
idofs=fem.locate_dofs_topological(space,0,imp_nodes)
bci=fem.dirichletbc(loading,idofs,space)
bcs=[bcb,bci]

### field

In [ ]:
%%px --local
disp=fem.Function(space,name='disp_fine')

### Generate Multipoint constraint

In [ ]:
%%px --local
mpc=core.generateMPC(j,space)

### Solve without mpc

In [ ]:
%%px --local
problem = petsc.LinearProblem(a_ufl, b_ufl,bcs=bcs,petsc_options=petsc_options)

In [ ]:
%%px --local
disp = problem.solve()

### Solve with mpc

In [ ]:
%%px --local
problemw = LinearProblem_mpc(a_ufl, b_ufl,mpc,bcs=bcs,petsc_options=petsc_options)


In [ ]:
%%px --local
dispw = problemw.solve()

### plot displacement solution at fine scale with or without MPC

In [ ]:
dispf_topology, dispf_cell_types, dispf_geometry = plot.vtk_mesh(space)
dispf_grid = pv.UnstructuredGrid(dispf_topology, dispf_cell_types, dispf_geometry)
dispf_gridw = pv.UnstructuredGrid(dispf_topology, dispf_cell_types, dispf_geometry)
#print(dispf_grid)
values = np.zeros((dispf_grid.GetNumberOfPoints(), 3))
values[:, :dim] = disp.x.array.reshape(dispf_grid.GetNumberOfPoints(), dim)
valuesw = np.zeros((dispf_gridw.GetNumberOfPoints(), 3))
valuesw[:, :dim] = dispw.x.array.reshape(dispf_grid.GetNumberOfPoints(), dim)
dispf_grid["disp"] =values
dispf_gridw["disp"] =valuesw
#dispf_grid.set_active_vectors("disp")
warpedf = dispf_grid.warp_by_vector("disp", factor=120)
warpedf.set_active_vectors("disp")
warpedfw = dispf_gridw.warp_by_vector("disp", factor=120)
warpedfw.set_active_vectors("disp")
plt=pv.Plotter(shape=(1, 2))
plt.subplot(0, 0)
plt.add_title("without MPC\nseq",font_size=15)
plt.add_mesh(warpedf,show_edges=True, show_scalar_bar=True)
plt.camera_position="xy"
plt.subplot(0, 1)
plt.add_title("with MPC\nseq",font_size=15)
plt.add_mesh(warpedfw,show_edges=True, show_scalar_bar=True)
plt.camera_position="xy"
plt.show()

In [ ]:
%%px
warpf=util.root_field(disp,100)
warpfm=util.root_field(dispw,100)
if cdomain.comm.rank == 0:
    plt=pv.Plotter(shape=(1, 2))
    plt.subplot(0, 0)
    plt.add_title("without MPC\npara",font_size=15)
    plt.add_mesh(warpf,**pvopt)
    plt.camera_position="xy"
    plt.subplot(0, 1)
    plt.add_title("with MPC\npara",font_size=15)
    plt.add_mesh(warpfm,**pvopt)
    plt.camera_position="xy"
    plt.show()

## Twoscale approach

In [ ]:
%%px
nested_strategy=core.useNest()

In [ ]:
%%px --target=0
if nested_strategy:
    print("Use nested strategy")
else:
    print("Use monolithic strategy")

### Enriched space

In [ ]:
%%px --local

el_mixed = basix.ufl.mixed_element([cdomain.ufl_domain().ufl_coordinate_element(), cdomain.ufl_domain().ufl_coordinate_element()])
coarse_enriched_space = fem.functionspace(cdomain, el_mixed)

std_=coarse_enriched_space.sub(0)
std, std_to_mix=std_.collapse()
enr_=coarse_enriched_space.sub(1)
enr, enr_to_mix=enr_.collapse()
std_to_mix_np=np.array(std_to_mix,dtype=int)

disp_ce=fem.Function(coarse_enriched_space,name='disp_coarse_enriched')


### Solve coarse non enriched problem (standard part) to init two scale loop

### Formulation

In [ ]:
%%px --local

muc=fem.Constant(cdomain,E / (2.0 * (1.0 + nu)))
lmbdac = fem.Constant(cdomain,E * nu / ((1.0 + nu) * (1.0 - 2.0 * nu)))
ucs = ufl.TrialFunction(std)
vcs = ufl.TestFunction(std)
def sigmac(strain): 
    return (1-dc)*(2.0*muc*strain + lmbdac*ufl.tr(strain)*ufl.Identity(dim))
acs=ufl.inner(sigmac(eps(ucs)), eps(vcs)) * ufl.dx
fcs=fem.Function(std,name='load')
fcs.x.array[:]=0.
bcst=ufl.dot(fcs,vcs)*ufl.dx

#### Boundary condition

In [ ]:
%%px
bc_nodesc=mesh.locate_entities(cdomain, 0, lambda x: np.isclose(x[0], 0.))
bdofscs=fem.locate_dofs_topological(std,0,bc_nodesc)
bcbcs=fem.dirichletbc(clamp,bdofscs,std)
imp_nodesc=mesh.locate_entities(cdomain, 0, lambda x: np.isclose(x[0], L))
idofscs=fem.locate_dofs_topological(std,0,imp_nodesc)
bcics=fem.dirichletbc(loading,idofscs,std)
bcscs=[bcbcs,bcics]

#### Resolution

In [ ]:
%%px
problemcs = petsc.LinearProblem(acs, bcst,bcs=bcscs,petsc_options=petsc_options)
dispcs = problemcs.solve()

In [ ]:
%%px
warpcs=util.root_field(dispcs,100)
if cdomain.comm.rank == 0:
    plt=pv.Plotter()
    plt.add_mesh(warpcs,**pvopt)
    plt.add_title("Coarse non enriched",font_size=15)
    plt.camera_position="xy"
    plt.show()

#### Store coarse standard in coarse enriched field

In [ ]:
%%px 
if nested_strategy:
    disp_ce=dispcs
else:
    disp_ce.x.array[std_to_mix_np]=dispcs.x.array
#print(disp_ce.x.array)

### Twoscale loop

#### Chose an enriched function

In [ ]:
%%px --local
enriched_shift=core.generateEnrichedShiftFunction(disp)

#### resolution with linearBasicLoop (i.e. scale loop resolution)

Dirichlet boundary condition must be defined on mixed space and merged into one Dirichlet BC object

In [ ]:
%%px 
if nested_strategy:
    BC_ce=bcscs
else:
    bc_nodesc=mesh.locate_entities(cdomain, 0, lambda x: np.isclose(x[0], 0.))
    imp_nodesc=mesh.locate_entities(cdomain, 0, lambda x: np.isclose(x[0], L))
    clamp_std_dofs=fem.locate_dofs_topological(std_,0,bc_nodesc)
    all_BC_values=fem.Function(coarse_enriched_space)
    imp_std_dof_x=fem.locate_dofs_topological(std.sub(0),0,imp_nodesc)
    imp_std_dof_x_ce=std_to_mix_np[imp_std_dof_x]
    if D1d:
        all_BC_dof=np.concat((clamp_std_dofs,imp_std_dof_x_ce))
        all_BC_values.x.array[imp_std_dof_x_ce]=imp
    else:
        if D2d:
            imp_std_dof_y=fem.locate_dofs_topological(std.sub(1),0,imp_nodesc)
            imp_std_dof_y_ce=std_to_mix_np[imp_std_dof_y]
            all_BC_dof=np.concat((clamp_std_dofs,imp_std_dof_x_ce,imp_std_dof_y_ce))
            all_BC_values.x.array[imp_std_dof_x_ce]=imp
        else:
            imp_std_dof_y=fem.locate_dofs_topological(std.sub(1),0,imp_nodesc)
            imp_std_dof_y_ce=std_to_mix_np[imp_std_dof_y]
            imp_std_dof_z=fem.locate_dofs_topological(std.sub(2),0,imp_nodesc)
            imp_std_dof_z_ce=std_to_mix_np[imp_std_dof_z]
            all_BC_dof=np.concat((clamp_std_dofs,imp_std_dof_x_ce,imp_std_dof_y_ce,imp_std_dof_z_ce))
            all_BC_values.x.array[imp_std_dof_x_ce]=imp
    all_BC_dof.sort()
    BC_ce=[fem.dirichletbc(all_BC_values,all_BC_dof)]


System are created and TS resolution is done:

In [ ]:
%%px 
[A,AD,BND,BD]=util.createFineScaleSytems(a_ufl,b_ufl,bcs=bcs,MPC=mpc)
[dispf, r,nm, it,hrb,hrr]=linear.linearBasicLoop(j,dispw.function_space,A,AD,BND,BD,enriched_shift,disp_ce,mpc,BC_ce,itmax,epsr)

Solutions with direct solver and TS solver are plotted bellow with theire differences

In [ ]:
%%px
warpts=util.root_field(dispf,100)
#warpfm=util.root_field(dispw,100)
if fdomain.comm.rank == 0:
    plt=pv.Plotter(shape=(2, 1))
    plt.subplot(0, 0)
    plt.add_mesh(warpts,**pvopt)
    plt.add_title("TS solution",font_size=14)
    plt.camera_position="xy"
    plt.subplot(1, 0)
    plt.add_mesh(warpfm,**pvopt)
    plt.add_title("fine scale solution",font_size=14)
    plt.camera_position="xy"
    plt.show()
ddisp=fem.Function(space,name='diff_disp_fine')
ddisp.x.array[:]=dispw.x.array-dispf.x.array
warpds=util.root_field(ddisp,1.e7,1)
if fdomain.comm.rank == 0:
    plt=pv.Plotter()
    plt.add_mesh(warpds,show_edges=False, show_scalar_bar=True, log_scale=True, copy_mesh=True)
    #plt.camera_position="xy"
    plt.show()


The following residual evolution history are plotted bellow:
* TS residual fine system relative to rhs $\frac{|AD.S_{ts_i}-BD|}{|BD|}$
* TS residual fine system relative to first residual $\frac{|AD.S_{ts_i}-BD|}{|AD.S_{ts_0}-BD|}$
* TS relative solution difference evolution $\frac{|S_{ts_i}-S_{ts_{i-1}})|}{|S_{ts_0}|}$


In [ ]:
%%px
[dispf3, r, it,pm,cm,hrb2,hrr2,hds2]=linear.linearBasicLoopVideo("pure_discontinuous.gif",100.,j,dispw.function_space,A,AD,BND,BD,enriched_shift,disp_ce,mpc,BC_ce,itmax,epsr,camera="xy")


In [ ]:
import matplotlib.pyplot as mplt
mplt.plot(view['hrb'][0],'o-',label='Residual relative to rhs')
mplt.plot(view['hrr'][0],'o-',label='Residual relative to first residual')
mplt.plot(view['hds2'][0],'*-',label='Relative TS displacement')
mplt.legend()
mplt.yscale('log')
mplt.xlabel('TS iterations')
mplt.ylabel('Criterion')

The  TS solver solution iterations are :
```{image} pure_discontinuous.gif
:width: 600px
:align: center
```


In [ ]:
%%px
problemw.solver.view()
print(problemw.solver.getIterationNumber())

In [ ]:
%%px --target=0
if nested_strategy:
    print ("With the nested strategy, convergence is not reached either with TS used as solver or as preconditioner.\nIt is certainly related to the iterative solver at the coarse level, perturbed by poor conditioning.\nWith monolithic approach, coarse solver is a LU direct solver and thus is less perturbed.\nClearly, this test case would be better if no coarse enriched nodes exist, and only extra enrichment nodes exist.\nIn this case, no patch would be close to rigid body mode movement...")